In [1]:
import os, json
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma

DOCS_DIR = "./docs"
DB_DIR = "../artifacts/chroma_db"

C:\Users\Mina_\AppData\Local\Temp\ipykernel_31944\799167495.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
docs = []
for fname in os.listdir(DOCS_DIR):
    if fname.endswith(".pdf"):
        pages = PyPDFLoader(os.path.join(DOCS_DIR, fname)).load()   # one Document per page
        docs.extend(pages)
        print(f"{fname}: {len(pages)} pages")

print("Total pages:", len(docs))

Credit discrimination is illegal _ Consumer Financial Protection Bureau.pdf: 3 pages
How to Improve Your Credit Score Fast.pdf: 12 pages
Personal Loan Requirements to Know Before You Apply.pdf: 13 pages
What Are the Different Types of Loans_ _ Wells Fargo.pdf: 7 pages
What Credit Score Is Needed for a Personal Loan_.pdf: 9 pages
What do I need to know about consolidating my credit card debt_ _ Consumer Financial Protection Bureau.pdf: 4 pages
What is a debt-to-income ratio_ _ Consumer Financial Protection Bureau.pdf: 2 pages
What is creditworthiness_ _ U.S. Bank.pdf: 8 pages
Total pages: 58


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print("Chunks:", len(chunks))
print(chunks[0].page_content[:300])
print(chunks[0].metadata)         

Chunks: 140
KNOW YOUR RIGHTS
Credit discrimination is illegal
Under federal law, lenders are not
allowed to discriminate against you.
When you apply for credit or borrow
money, it is against the law to
discriminate because of:
Race
Color
Religion
National origin
Sex
Marital status
Age
Receiving money from publi
{'producer': 'Skia/PDF m158', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/158.0.0.0 Safari/537.36', 'creationdate': '2026-11-26T11:58:42+00:00', 'title': 'Credit discrimination is illegal | Consumer Financial Protection Bureau', 'moddate': '2026-11-26T11:58:42+00:00', 'source': './docs\\Credit discrimination is illegal _ Consumer Financial Protection Bureau.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


In [ ]:
embedder = OllamaEmbeddings(model="nomic-embed-text", num_gpu=0)

REBUILD = True        

store = Chroma(collection_name="lending_docs", embedding_function=embedder, persist_directory=DB_DIR)

if REBUILD:
    store.delete_collection()                      
    store = Chroma(collection_name="lending_docs", embedding_function=embedder, persist_directory=DB_DIR)

if store._collection.count() == 0:                 
    store.add_documents(chunks)
    print(f"Stored {len(chunks)} chunks in Chroma.")
else:
    print("Existing Chroma database loaded:", store._collection.count(), "chunks")

Stored 140 chunks in Chroma.


In [5]:
def search_lending_documents(query: str, k: int = 4) -> str:
    """Return the k most relevant passages from the lending documents as JSON."""
    if not query or not query.strip():
        return json.dumps({"error": "Empty query. Please provide a question."})
    try:
        retriever = store.as_retriever(search_kwargs={"k": k})
        relevant_docs = retriever.invoke(query)
        passages = [{
            "text": doc.page_content,
            "source": os.path.basename(doc.metadata.get("source", "")),
            "page": doc.metadata.get("page", 0) + 1,        # PyPDFLoader counts pages from 0
        } for doc in relevant_docs]
        return json.dumps({"query": query, "passages": passages}, ensure_ascii=False, indent=2)
    except Exception as e:
        return json.dumps({"error": f"Search failed: {e}"})

In [ ]:
print(search_lending_documents("What credit score do I need for a personal loan?"))
print(search_lending_documents(""))          

{
  "query": "What credit score do I need for a personal loan?",
  "passages": [
    {
      "text": "Advertiser Disclosure \n / LOANS / PERSONAL LOANS\nWhat Credit Score Is Needed\nfor a Personal Loan?\nBy Ben Luthi •\n Reviewed by Peggy Bennett\n5 min read • Sep 27, 2026\nQUICK ANSWER\nYou generally need credit scores of 580 or higher to\nqualify for a personal loan. You'll typically need\nscores in the 700s to qualify with favorable terms.\nComparing loan offers can help you understand\nwhich loans you might qualify for.\nIf you have time to improve your credit before\napplying for a personal loan, it could save you money.\nIN THIS ARTICLE\nHow Does Your Credit Score Impact Your Personal Loan Application?\n\nSign in \n11/25/26, 5:42 PM What Credit Score Is Needed for a Personal Loan?\nhttps://www.experian.com/blogs/ask-experian/what-credit-score-is-needed-for-a-personal-loan/ 1/11",
      "source": "What Credit Score Is Needed for a Personal Loan_.pdf",
      "page": 1
    },
  

In [7]:
question = "What is a debt-to-income ratio and how is it calculated?"
result = json.loads(search_lending_documents(question, k=3))
context = "\n\n".join(p["text"] for p in result["passages"])

llm = ChatOllama(model="qwen3.5:4b", reasoning=False , num_ctx=4096, num_gpu=0)

prompt = f"""Answer using ONLY the context below.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {question}
Answer:"""

print(llm.invoke(prompt).content)
print("\nSources:", {(p["source"], p["page"]) for p in result["passages"]})

A debt-to-income ratio (DTI) is a number used by lenders to measure your ability to manage monthly payments to repay the money you plan to borrow. It is calculated by adding up all your monthly debt payments and dividing them by your gross monthly income, which is generally the amount of money earned before taxes and other deductions are taken out.

Sources: {('What is a debt-to-income ratio_ _ Consumer Financial Protection Bureau.pdf', 1), ('What is a debt-to-income ratio_ _ Consumer Financial Protection Bureau.pdf', 2)}


In [8]:
question = "What is the interest rate for a mortgage in Egypt?"
result = json.loads(search_lending_documents(question, k=3))
context = "\n\n".join(p["text"] for p in result["passages"])
print(llm.invoke(f"""Answer using ONLY the context below.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {question}
Answer:""").content)

I don't know. The provided context discusses debt consolidation loans, home equity loans, personal loans, and their terms (such as teaser rates, origination fees, fixed vs. variable interest rates), but it does not mention mortgages in Egypt or any specific interest rate for them.
